# Robustness of Accessibility
## Notebook 3/4 - Draw samples, calculate distances and scores

In [ ]:
from __future__ import annotations

import datetime
import time
import json
from shapely.geometry import shape
import osmnx as ox
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import Point
import folium

from pathlib import Path
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_hazard_data_path,
    get_roa_inputs_path,
    get_roa_outputs_path,
)

# ox.config(use_cache=True, log_console=True)
ox.settings.log_console = False
ox.settings.use_cache = True

ox.__version__

In [ ]:
event = RoaNotebookConfig.event

inputs_dir: Path = get_roa_inputs_path()
cache_dir: Path = get_roa_cache_path()
output_dir: Path = get_roa_outputs_path()
hazard_data_path: Path = get_roa_hazard_data_path(event=event)

In [ ]:
# set globals:
start_time_global = time.time()
place_name: str = RoaNotebookConfig.place_name
undirected_graph: bool = RoaNotebookConfig.undirected_graph

show_all_graph = False

### Create  basic graph of Trier

In [ ]:
# load networks for calculation
road_network = ox.load_graphml(cache_dir / f"network/drive_graph_{place_name}.graphml")
road_network_remaining = ox.load_graphml(cache_dir / f"network/drive_graph_remaining_{place_name}.graphml")

#### Make graph undirected since services might ignore one way roads

In [ ]:
# make graph undirected to allow services to move in both directions
if undirected_graph:
    road_network = road_network.to_undirected()
    road_network_remaining = road_network_remaining.to_undirected()

#### Optionally add speed information to calculate travel time of edges

In [ ]:
# impute speed on all edges missing data
road_network_remaining = ox.add_edge_speeds(road_network_remaining)
road_network = ox.add_edge_speeds(road_network)

# calculate travel time (seconds) for all edges
road_network_remaining = ox.add_edge_travel_times(road_network_remaining)
road_network = ox.add_edge_travel_times(road_network)

In [ ]:
## 1. convert the graph to GeoDataFrame
road_network_remaining_nodes, road_network_remaining_edges = ox.graph_to_gdfs(road_network_remaining)
road_network_nodes, road_network_edges = ox.graph_to_gdfs(road_network)

### Load flooding data and select scenario

In [ ]:
# Import shapes of flooded areas
hazard_area = gpd.read_file(hazard_data_path)
hazrd_area_geoms = hazard_area.geometry

# Combine all shapes of the flooded are into one to make it easier to work with
hazard_area_multipolygon_df = gpd.GeoSeries(unary_union(hazrd_area_geoms))
hazard_area_multipolygon = hazard_area_multipolygon_df[0]

In [ ]:
hospitals = gpd.read_file(cache_dir / f"services/hospitals_{place_name}.geojson")
fire_stations = gpd.read_file(cache_dir / f"services/fire_stations_{place_name}.geojson")
administrative_districts = gpd.read_file(cache_dir / f"services/administrative_districts_{place_name}.geojson")

In [ ]:
with open(cache_dir / f"services/boundary_geom_{place_name}.geojson", "r") as f:
    # as_string = f.read()
    geom_dict = json.load(f)  # → a plain GeoJSON *geometry* dict
boundary_geom = shape(geom_dict)  # → Shapely Polygon or MultiPolygon
boundary_geom

In [ ]:
# refactor using default functon
from css_geodata_service.robustness_of_accessibility.robustness_of_accessibility import (
    get_nearest_node_id,
    get_node_point_from_node_id,
)


def prepare_services(services: gpd.GeoDataFrame, drive_service_graph) -> gpd.GeoDataFrame:
    print("Prepare services")
    services = services.copy(deep=True)
    services["nearest_node_id"] = services["position"].apply(
        lambda point: get_nearest_node_id(point=point, graph=drive_service_graph)
    )

    street_network_nodes, _ = ox.graph_to_gdfs(drive_service_graph)
    services["nearest_node_point"] = services["nearest_node_id"].apply(
        lambda node_id: get_node_point_from_node_id(nodes=street_network_nodes, node_id=node_id)
    )
    return services

In [ ]:
fire_stations["position"] = fire_stations["geometry"].centroid
new_fire_stations = prepare_services(fire_stations, road_network)

In [ ]:
# filter hospitals that are relevant i.e. have a emergency reception
hospitals = hospitals.loc[hospitals["emergency"] == "yes"]
hospitals

In [ ]:
# since osm data is often not perfect and we might have special knowledge we can manually adapt the data to make it more realistic
# we exclude Klinikum Mutterhaus der Borromäerinnen Nord since it actually has no emergency reception
hospitals = hospitals[hospitals["name"] != "Klinikum Mutterhaus der Borromäerinnen Nord"]

In [ ]:
hospitals.reset_index(inplace=True, drop=True)
hospitals

### Disclaimer about coordinates:
- Shapely uses: (lon, lat)
- Folium / leaflet uses (lat, lon)

### Fix hospital emergency access location 
- Hospitals are big buildings with a sometimes lot of adjecent roads
- Hospitals are big buildings and usually not part of the road network itself
- Calculating a route to a hospital via the nearest node to the centroid of the geometry may be unrealsitic 
- The following section is used to manually select / define the nearst access / node for every hospital in order to ensure that routes are realistic.
- identified node ids are used later for routing

In [ ]:
hospitals.loc[:, "position"] = None
hospitals

In [ ]:
hospitals.loc[hospitals["name"] == "Klinikum Mutterhaus der Borromäerinnen Trier", "position"] = Point(
    6.632108, 49.754674
)
hospitals.loc[(hospitals["name"] == "Krankenhaus der Barmherzigen Brüder Trier"), "position"] = Point(
    6.640020, 49.761562
)

In [ ]:
hospitals["nearest_node_id"] = hospitals["position"].apply(
    lambda point: get_nearest_node_id(point=point, graph=road_network)
)

In [ ]:
hospitals["nearest_node_id"] = hospitals["position"].apply(
    lambda point: get_nearest_node_id(point=point, graph=road_network)
)
hospital_nodes_df = road_network_remaining_nodes.loc[hospitals["nearest_node_id"]].reset_index()
# nodes_df.reset_index(inplace=True)
hospitals["nearest_node_point"] = hospital_nodes_df["geometry"]
hospitals.head()

In [ ]:
region_center = boundary_geom.centroid
print(f"x={region_center.x} y={region_center.y}")

### Visualization only (hospitals)

In [ ]:
from css_geodata_service.robustness_of_accessibility.utils.utils import (
    add_points_from_df_to_folium_map,
)


hospital_map = folium.Map(location=[region_center.y, region_center.x], zoom_start=13)

hospital_map = add_points_from_df_to_folium_map(df=hospitals, folium_map=hospital_map, column="position", text="Access")
hospital_map = add_points_from_df_to_folium_map(
    df=hospitals,
    folium_map=hospital_map,
    column="nearest_node_point",
    text="Nearest Node",
)
# hospital_map

### Fires Stations
- Fire Stations are much smaller compared to hospitals so we just use the centroid

In [ ]:
# set position for each fire station
fire_stations["position"] = fire_stations["geometry"].centroid

In [ ]:
fire_stations["nearest_node_id"] = fire_stations["position"].apply(
    lambda point: get_nearest_node_id(point=point, graph=road_network)
)
fire_station_nodes_df = road_network_nodes.loc[fire_stations["nearest_node_id"]].reset_index()
# nodes_df.reset_index(inplace=True)
fire_stations["nearest_node_point"] = fire_station_nodes_df["geometry"]

### Visualization only (fire stations)

In [ ]:
fire_station_map = folium.Map(location=[region_center.y, region_center.x], zoom_start=13, tiles="CartoDB positron")

fire_station_map = add_points_from_df_to_folium_map(
    df=fire_stations, folium_map=fire_station_map, column="position", text="Centroid"
)
fire_station_map = add_points_from_df_to_folium_map(
    df=fire_stations,
    folium_map=fire_station_map,
    column="nearest_node_point",
    text="Nearest Node",
)
# fire_station_map

### Exclude the boundaries that do not have a name or those that have more than one district (hierarchy)

In [ ]:
districts_filtered = administrative_districts.loc[administrative_districts["name"].notnull()]
print(f"districts_filtered with name = {len(districts_filtered)} all boundaries = {len(administrative_districts)}")

### Visualize the district boundaries in the area with their names

In [ ]:
combined_district_polygon = unary_union(districts_filtered.geometry)
gdf_combined_district_polygons = gpd.GeoDataFrame(geometry=[combined_district_polygon])

In [ ]:
from css_geodata_service.robustness_of_accessibility.robustness_of_accessibility import draw_sample

# TODO before drawing a sample we must ensure the buffer
samples = draw_sample(boundary_geom, road_network_nodes, sample_distance_in_meters=50)

In [ ]:
# filter samples to only those that are within a considered administrative adrea
print(f"number of (pre-filtered) samples = {len(samples)}")
samples["within_polygon"] = samples["geometry"].apply(lambda point: combined_district_polygon.contains(point))
df_node_samples_within_polygon = samples[samples["within_polygon"]].drop(labels="within_polygon", axis=1)
print(f"number of (filtered) samples = {len(df_node_samples_within_polygon)}")

In [ ]:
df_node_samples_within_polygon.plot(figsize=(10, 10))

In [ ]:
# filter samples for testing
sub_sample = False
if sub_sample:
    df_node_samples_within_polygon_sample = df_node_samples_within_polygon.sample(frac=0.01, random_state=42)
    print(
        f" original number of samples = {len(df_node_samples_within_polygon)}, sampled number = {len(df_node_samples_within_polygon_sample)}"
    )
    df_node_samples_within_polygon = df_node_samples_within_polygon_sample

In [ ]:
from css_geodata_service.robustness_of_accessibility.utils.models import NodeDetails
from css_geodata_service.robustness_of_accessibility.utils.utils import (
    calculate_routes,
    create_distance_df,
)

start = time.time()
print("Calculate RoA distances and scores")

poi = {"fire_stations": fire_stations, "hospitals": hospitals}
results: dict[str, dict] = {"normal": {}, "disrupted": {}}
for state in results.keys():
    s = time.time()
    network = None
    if state == "normal":
        network = road_network
    if state == "disrupted":
        network = road_network_remaining
    # calculate routes to services in both states
    node_details: list[NodeDetails] = calculate_routes(
        samples=samples, services=poi, street_network=network, state=state
    )
    print(f"{round((time.time() - s) / 60, 2)} minutes elapsed for {state} street network")
    distances = create_distance_df(node_details)
    # split up service types
    for service_type in distances["service_type"].unique():
        results[state][service_type] = distances[distances["service_type"] == service_type]
end = time.time()
print(f"Calculate RoA elapsed time: {end - start}")

In [ ]:
from css_geodata_service.robustness_of_accessibility.robustness_of_accessibility import calculate_roa_scores

scores = calculate_roa_scores(results)

### Write results to file

In [ ]:
now: datetime.datetime = datetime.datetime.now()

In [ ]:
for service_type in scores.keys():
    scores_for_service = scores[service_type]
    scores_for_service.to_csv(output_dir / f"scores_{service_type}_{now}.csv")
str(now)

In [ ]:
# Record the end time
end_time_global = time.time()

# Calculate the time elapsed between the two commands
elapsed_time_global = end_time_global - start_time_global

In [ ]:
print(f"{round(elapsed_time_global / 60, 2)} minutes elapsed for total process")